In [91]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
while project_root.name != 'olist-delivery-risk':
    if project_root == project_root.parent:
        raise FileNotFoundError("找不到项目根目录 olist-delivery-risk")  
    project_root = project_root.parent

delay_orders = pd.read_csv(project_root/'data'/'processed'/'orders_with_delay.csv')
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']
delay_orders[date_cols] = delay_orders[date_cols].apply(pd.to_datetime, errors="coerce")

print(delay_orders.dtypes)

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
item_count                              float64
product_amount                          float64
freight_amount                          float64
total_amount                            float64
customer_unique_id                          str
customer_zip_code_prefix                  int64
customer_city                               str
customer_state                              str
month                                       str
is_delayed                                int64
dtype: object


In [92]:
# 定义候选特征
candidate_features = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date",
    "customer_state",
    "customer_city",
    "customer_zip_code_prefix",
    "item_count",
    "product_amount",
    "freight_amount",
    "total_amount"
]

# 统计每个候选字段的缺失情况
missing_summary = pd.DataFrame({
     "missing_count":delay_orders[candidate_features].isna().sum(),
     "missing_rate":delay_orders[candidate_features].isna().mean()*100
})
missing_summary 

,missing_count,missing_rate
order_purchase_timestamp,0,0.0
order_approved_at,0,0.0
order_estimated_delivery_date,0,0.0
customer_state,0,0.0
customer_city,0,0.0
customer_zip_code_prefix,0,0.0
item_count,0,0.0
product_amount,0,0.0
freight_amount,0,0.0
total_amount,0,0.0


In [93]:
# 定义禁止作为模型输入的字段
forbidden_features = [
    'order_id',                      # 订单唯一标识
    'customer_id',                   # 客户订单标识
    'customer_unique_id',            # 客户身份标识
    'order_status',                  # 最终订单状态
    'order_delivered_carrier_date',  # 实际交付承运商时间
    'order_delivered_customer_date', # 实际送达时间
    'is_delayed'                     # 预测标签，不能作为输入特征
]

# 检查候选特征和禁用字段是否存在交集
leakage_check = set(candidate_features) & set(forbidden_features)

print('候选特征中的禁用字段:', leakage_check)

候选特征中的禁用字段: set()


In [94]:
valid = (
    delay_orders["order_status"].eq("delivered")
    & delay_orders["order_approved_at"].notna()
    & delay_orders["order_delivered_customer_date"].notna()
    & delay_orders["order_estimated_delivery_date"].notna()
)

expected_label = (
    delay_orders["order_delivered_customer_date"].dt.normalize()
    > delay_orders["order_estimated_delivery_date"].dt.normalize()
).astype(int)

print("不符合样本规则的数量：", (~valid).sum())

print(
    "延迟标签不一致的数量：",
    (delay_orders["is_delayed"] != expected_label).sum()
)

#不一致，则停止
assert (
    delay_orders.loc[valid, "is_delayed"]
    == expected_label.loc[valid]
).all(), "延迟标签与日历日期比较结果不一致"

不符合样本规则的数量： 0
延迟标签不一致的数量： 0


In [95]:
# 读取异常订单排除名单
excluded = pd.read_csv(
    project_root / "data/processed/excluded_orders.csv"
)

# 检查排除名单是否存在重复ID
assert excluded["order_id"].is_unique

# 有效建模样本
valid_orders = delay_orders.loc[valid].copy()

# 从有效样本中排除异常订单
ml_data = valid_orders.loc[
    ~valid_orders["order_id"].isin(excluded["order_id"])
].copy()

# 统计被排除的订单数量
removed_count = len(valid_orders) - len(ml_data)

print("原有效建模样本数：", len(valid_orders))
print("异常名单订单数：", len(excluded))
print("实际排除数量：", removed_count)
print("处理后建模样本数：", len(ml_data))

# 检查排除结果
assert not ml_data["order_id"].isin(
    excluded["order_id"]
).any()

# 查看处理后的延迟标签
print("\n标签分布：")
print(ml_data["is_delayed"].value_counts())

print("\n延迟率：")
print(ml_data["is_delayed"].mean() * 100)


原有效建模样本数： 96456
异常名单订单数： 84
实际排除数量： 84
处理后建模样本数： 96372

标签分布：
is_delayed
0    89839
1     6533
Name: count, dtype: int64

延迟率：
6.7789399410617195


In [96]:
# 下单时间特征
ml_data["purchase_month"] = (
    ml_data["order_purchase_timestamp"].dt.month
)

ml_data["purchase_weekday"] = (
    ml_data["order_purchase_timestamp"].dt.dayofweek
)

ml_data["purchase_hour"] = (
    ml_data["order_purchase_timestamp"].dt.hour
)

# 订单获批时间特征
ml_data["approval_hour"] = (
    ml_data["order_approved_at"].dt.hour
)

In [97]:
model_features = [
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "approval_hour",
    "customer_state",
    "customer_city",
    "item_count",
    "product_amount",
    "freight_amount"
]

# 确认选择的特征没有包含禁用字段
assert not (set(model_features) & set(forbidden_features))

# 构建特征矩阵 X
X = ml_data[model_features].copy()

# 构建目标标签 y
y = ml_data["is_delayed"].copy()

print("X 的形状：", X.shape)
print("y 的形状：", y.shape)

X 的形状： (96372, 9)
y 的形状： (96372,)


In [98]:
# 检查特征缺失情况
model_missing = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_rate": X.isna().mean() * 100
})

print(model_missing)

# 检查标签是否有缺失
print("标签缺失数：", y.isna().sum())

# 检查标签分布
print(y.value_counts())

# 检查延迟比例
print(y.value_counts(normalize=True))

# 检查样本数量是否一致
print("X和y行数一致：", len(X) == len(y))

# 检查城市类别数量
print("城市类别数量：", X["customer_city"].nunique())

                  missing_count  missing_rate
purchase_month                0           0.0
purchase_weekday              0           0.0
purchase_hour                 0           0.0
approval_hour                 0           0.0
customer_state                0           0.0
customer_city                 0           0.0
item_count                    0           0.0
product_amount                0           0.0
freight_amount                0           0.0
标签缺失数： 0
is_delayed
0    89839
1     6533
Name: count, dtype: int64
is_delayed
0    0.932211
1    0.067789
Name: proportion, dtype: float64
X和y行数一致： True
城市类别数量： 4085


In [99]:
model_dataset = ml_data[
     ["order_id", "order_approved_at"]
     + model_features
     + ["is_delayed"]
].copy()

#路径
output_path = (project_root/'data'/'processed'/'model_dataset.csv')

#保存
model_dataset.to_csv(output_path,index=False)
print("保存成功：", output_path)
print("数据形状：", model_dataset.shape)

保存成功： d:\各类文件\项目\olist-delivery-risk\data\processed\model_dataset.csv
数据形状： (96372, 12)


In [100]:
# 最终数据验证

print("===== 最终建模数据检查 =====")

print("建模数据行数：", len(model_dataset))
print("建模数据列数：", model_dataset.shape[1])

print("\n订单ID是否唯一：")
print(model_dataset["order_id"].is_unique)

print("\n是否仍有需要排除的异常订单：")
print(
    model_dataset["order_id"]
    .isin(excluded["order_id"]).sum()
)

print("\n特征缺失值：")
print(model_dataset[model_features].isna().sum())

print("\n最终标签分布：")
print(model_dataset["is_delayed"].value_counts())

print("\n最终延迟率：")
print(
    round(model_dataset["is_delayed"].mean() * 100, 4),
    "%"
)

# 自动检查
assert model_dataset["order_id"].is_unique
assert not model_dataset["order_id"].isin(
    excluded["order_id"]
).any()
assert model_dataset["is_delayed"].isin([0, 1]).all()
assert model_dataset["order_approved_at"].notna().all()
assert model_dataset[model_features].notna().all().all()

# 检查保存到磁盘后的数据
saved_model = pd.read_csv(output_path)

assert saved_model.shape == model_dataset.shape
assert saved_model["order_id"].equals(
    model_dataset["order_id"].reset_index(drop=True)
)

print("\n最终检查通过！")


===== 最终建模数据检查 =====
建模数据行数： 96372
建模数据列数： 12

订单ID是否唯一：
True

是否仍有需要排除的异常订单：
0

特征缺失值：
purchase_month      0
purchase_weekday    0
purchase_hour       0
approval_hour       0
customer_state      0
customer_city       0
item_count          0
product_amount      0
freight_amount      0
dtype: int64

最终标签分布：
is_delayed
0    89839
1     6533
Name: count, dtype: int64

最终延迟率：
6.7789 %

最终检查通过！


# 第 8 天：预测任务定义与特征工程

## 1. 预测任务定义

本阶段基于 `orders_with_delay.csv` 构建订单配送延迟预测任务，并在第 9 天异常数据处理完成后更新建模样本。

**预测目标：**

在订单获批时，根据已知的订单、商品及客户信息，预测订单最终是否发生配送延迟。

该任务属于二分类问题，目标变量为 `is_delayed`：

- `is_delayed = 0`：订单未延迟。
- `is_delayed = 1`：订单发生延迟。

**延迟标签定义：**

采用日历日期比较规则。只有实际送达日期晚于预计送达日期时，才判定为延迟。

同一天送达的订单，无论具体送达时刻如何，均不判定为延迟。

---

## 2. 候选特征与数据泄漏控制

### 2.1 候选特征

初步考虑订单时间、客户地区和商品信息：

| 特征类别 | 候选字段 |
|---|---|
| 时间信息 | `order_purchase_timestamp`、`order_approved_at`、`order_estimated_delivery_date` |
| 地区信息 | `customer_state`、`customer_city`、`customer_zip_code_prefix` |
| 商品信息 | `item_count`、`product_amount`、`freight_amount`、`total_amount` |

通过缺失值统计了解各候选字段的数据完整性，并根据预测时点筛选最终特征。

### 2.2 禁止输入模型的字段

为减少数据泄漏，明确以下字段不得直接进入模型输入矩阵：

- `order_id`：订单唯一标识。
- `customer_id`：客户订单标识。
- `customer_unique_id`：客户身份标识。
- `order_status`：最终订单状态。
- `order_delivered_carrier_date`：实际交付承运商时间。
- `order_delivered_customer_date`：实际送达时间。
- `is_delayed`：预测目标变量。

使用集合交集检查，验证候选特征和最终模型特征中是否包含禁止字段。

此外，`order_estimated_delivery_date` 暂未纳入最终特征，需要进一步确认其在订单获批时是否已知。

商品金额、运费等字段的使用基于它们在订单获批时已经确定的业务假设。

---

## 3. 建模样本筛选与延迟标签验证

### 3.1 基本建模样本筛选

首先按照以下条件筛选有效订单：

1. 订单状态为 `delivered`。
2. 订单获批时间不缺失。
3. 实际送达时间不缺失。
4. 预计送达时间不缺失。

满足以上条件的订单组成 `valid_orders`。

筛选结果：

**原有效建模样本数量：96,456 条。**

### 3.2 延迟标签一致性检查

根据实际送达日期与预计送达日期重新计算 `expected_label`，并与已有 `is_delayed` 进行比较。

使用 `.dt.normalize()` 去除时间中的时分秒，确保判断遵循日历日期比较规则。

对有效建模样本使用 `assert` 进行一致性验证，避免标签定义不一致的数据继续进入模型。

---

## 4. 异常数据排除

### 4.1 异常处理规则

根据第 9 天的时间异常检查，最终采用以下处理策略：

| 异常类型 | 检查发现数量 | 处理方式 |
|---|---:|---|
| 实际送达早于获批 | 61 | 排除 |
| 实际送达早于物流交接 | 23 | 排除 |
| 其他物流交接早于获批的订单 | 1,298 | 暂时保留并记录 |
| 极端金额订单 | 存在 | 未发现明确计算错误，暂时保留 |

两类需要排除的异常通过 `order_id` 汇总并去重，最终形成 84 条订单的排除名单。

该名单保存于：

`data/processed/excluded_orders.csv`

### 4.2 排除结果

在原有效建模样本基础上，根据异常名单过滤订单。

| 指标 | 数量 |
|---|---:|
| 原有效建模样本 | 96,456 |
| 异常排除名单订单 | 84 |
| 实际排除订单 | 84 |
| **处理后建模样本** | **96,372** |

排除的 84 条订单中：

- 未延迟订单：83 条。
- 延迟订单：1 条。

代码通过断言确认，最终建模数据中不再包含排除名单中的订单。

该过程仅更新用于机器学习的建模数据，不直接修改原始订单数据。

---

## 5. 时间特征工程

从原始时间字段中提取以下 4 个数值特征：

| 特征 | 来源 | 含义 |
|---|---|---|
| `purchase_month` | 下单时间 | 下单月份 |
| `purchase_weekday` | 下单时间 | 下单星期，0 为星期一 |
| `purchase_hour` | 下单时间 | 下单小时 |
| `approval_hour` | 获批时间 | 获批小时 |

这些特征用于表示订单产生的时间信息及可能存在的周期性差异。

原始时间戳不直接作为第一版模型的输入特征。

---

## 6. 第一版模型特征

最终选择 9 个特征作为模型输入：

| 特征类别 | 字段 | 数量 |
|---|---|---:|
| 时间信息 | `purchase_month`、`purchase_weekday`、`purchase_hour`、`approval_hour` | 4 |
| 地区信息 | `customer_state`、`customer_city` | 2 |
| 商品信息 | `item_count`、`product_amount`、`freight_amount` | 3 |
| **合计** | | **9** |

使用上述特征构建特征矩阵 `X`，使用 `is_delayed` 构建目标变量 `y`。

实际运行结果：

- `X.shape = (96372, 9)`
- `y.shape = (96372,)`
- `X` 和 `y` 样本数量一致：`True`

---

## 7. 最终建模数据检查

### 7.1 数据规模

| 指标 | 结果 |
|---|---:|
| 最终建模样本数量 | 96,372 |
| 输入特征数量 | 9 |
| 目标标签数量 | 96,372 |
| 最终数据集字段数量 | 12 |
| 订单 ID 是否唯一 | True |

最终数据集的 12 个字段包括：

- `order_id`
- `order_approved_at`
- 9 个模型输入特征
- `is_delayed`

其中，`order_id` 用于追踪订单，`order_approved_at` 用于后续按时间划分数据集，二者不直接进入模型输入矩阵。

### 7.2 特征缺失值检查

对最终 9 个输入特征统计缺失值：

| 特征 | 缺失数量 | 缺失比例 |
|---|---:|---:|
| `purchase_month` | 0 | 0.00% |
| `purchase_weekday` | 0 | 0.00% |
| `purchase_hour` | 0 | 0.00% |
| `approval_hour` | 0 | 0.00% |
| `customer_state` | 0 | 0.00% |
| `customer_city` | 0 | 0.00% |
| `item_count` | 0 | 0.00% |
| `product_amount` | 0 | 0.00% |
| `freight_amount` | 0 | 0.00% |

**检查结论：**

最终 9 个模型特征均不存在缺失值。

因此，后续可以直接进行数据划分，但仍需根据模型要求进行类别编码和数值特征预处理。

### 7.3 延迟标签分布

| 标签 | 含义 | 订单数量 | 占比 |
|---|---|---:|---:|
| 0 | 未延迟 | 89,839 | 93.2211% |
| 1 | 延迟 | 6,533 | 6.7789% |
| **合计** | | **96,372** | **100.00%** |

整体延迟率：

\[
\text{Delay Rate}
=\frac{6533}{96372}\times100\%
\approx6.7789\%
\]

**分析结论：**

1. 未延迟订单占绝大多数，约为 93.22%。
2. 延迟订单占比约为 6.78%，存在明显的类别不平衡。
3. 异常订单处理前后，整体延迟率变化较小。
4. 后续模型评估不宜只依赖准确率（Accuracy），还应关注 Average Precision（AP）、Precision 和 Recall 等指标。

### 7.4 城市类别检查

最终建模样本中：

**`customer_city` 不同取值数量：4,085。**

客户城市属于高基数类别特征。

如果直接使用 One-Hot 编码，可能产生大量编码列。

因此，后续建模时考虑：

- 使用 `OneHotEncoder` 进行类别编码。
- 正确处理验证集和测试集中未见过的城市。
- 根据训练集的类别频数决定是否合并低频城市。
- 比较仅使用州与同时使用州、城市两种特征方案。

所有需要学习数据分布的预处理操作都应仅在训练集上拟合。

---

## 8. 最终建模数据保存与验证

最终建模数据保存为：

`data/processed/model_dataset.csv`

保存后重新读取文件，检查磁盘文件与内存数据是否一致。

最终执行的检查包括：

1. 订单 ID 唯一。
2. 异常排除名单中的订单残留数量为 0。
3. 延迟标签只包含 0 和 1。
4. 获批时间不存在缺失值。
5. 9 个模型特征均不存在缺失值。
6. 保存文件的形状与内存中的数据一致。
7. 保存文件的订单 ID 顺序与内存数据一致。

程序最终输出：

**`最终检查通过！`**

说明当前生成的建模数据满足代码中定义的质量检查要求。

---

## 9. 当前数据局限

尽管已经完成基本筛选及异常订单处理，当前建模数据仍存在一些局限：

1. **样本选择限制：** 当前仅包含满足条件的已送达订单，可能与实际预测时面对的全部新订单存在差异。
2. **预测时点假设：** 商品金额、运费等信息在订单获批时是否确定，仍需结合业务情况确认。
3. **异常数据限制：** 本次仅排除有明确规则的时间异常订单，其余未确认错误的记录仍可能存在。
4. **类别稀疏问题：** 城市具有较多类别，后续需采取合适的编码方式。
5. **类别不平衡问题：** 延迟订单比例较低，模型评估需要关注少数类识别能力。
6. **时间泄漏风险：** 虽然已经排除未来事件字段，但尚需在时间划分时检查训练样本的标签是否已在相应时间点可观察到。

---

## 10. 本阶段总结

第 8 天完成了预测任务定义及第一版特征工程，第 9 天进一步完成异常检查并更新建模数据。

主要成果：

1. 明确订单获批时预测配送延迟的二分类任务。
2. 统一采用日历日期规则定义延迟标签。
3. 确定候选特征及禁止输入模型的字段。
4. 筛选有效建模样本并验证标签一致性。
5. 根据异常排除名单删除 84 条订单。
6. 提取 4 个时间特征。
7. 确定 9 个模型输入特征。
8. 构建特征矩阵 `X` 和目标变量 `y`。
9. 检查特征缺失、订单唯一性及标签分布。
10. 保存包含 96,372 条订单、12 个字段的最终建模数据。
11. 完成建模数据的保存与完整性验证。

## 11. 下一阶段工作：

第 10 天将基于更新后的 `model_dataset.csv` 开展数据划分和基线模型训练。

主要工作：

1. 根据 `order_approved_at` 按时间顺序划分训练集、验证集和测试集。
2. 检查各数据集的样本数量及延迟率。
3. 检查标签在训练时点是否已经可观察到，避免时间泄漏。
4. 建立 `DummyClassifier` 基线模型。
5. 使用 Average Precision、Precision、Recall 等指标评估基线表现。
6. 为后续逻辑回归模型提供比较基准。